# 모서리 깎기(chamfer) 2대 시험 — v2

앞선 v1에서 두 가지를 고쳤습니다.

1. **경사를 두께의 절반에만 넣습니다.** v1은 두께 전체(5 mm)를 깎아서 뒤쪽 **아랫면 모서리**까지 없앴고, 그 자리에 있는 발톱(cleat)이 바닥을 못 잡아 오히려 뒤로 밀렸습니다(전진 −0.67 cm). v2는 뒤끝은 *윗면 절반*만, 앞끝은 *아랫면 절반*만 깎습니다.
2. **판정 기준을 고쳤습니다.** v1은 상대 등 높이를 초기화 직후(로봇이 아직 2 mm 떠 있을 때) 한 번만 재서 7 mm로 잘못 잡았고, 몸통 중심 높이로 판정해 몸을 접어 들리기만 해도 "올라탐"이 됐습니다. v2는 매 스텝 상대의 현재 등 높이를 다시 재고, 앞 끝 좌표와 가운데 조각 밑면으로 판정합니다.

경사 **없음 / 5 mm / 10 mm** 세 가지를, 출발 위치 4가지로 각각 30초씩 돌립니다(총 12회, 5분 안팎).

`BARISimulation` 폴더에 두고 **`barisimulation` 커널**로 위에서부터 실행하세요.

In [ ]:
# 1) 준비: 모서리 깎기(chamfer)를 XML 후처리로 넣는다. 저장소 코드는 건드리지 않는다.
#    v2: 경사를 두께의 "위 절반"(뒤끝)과 "아래 절반"(앞끝)에만 넣는다.
#        v1처럼 두께 전체를 깎으면 뒤쪽 아랫면 모서리가 사라져 발톱(cleat) 접지력을 잃는다.
import math, gc, time, dataclasses
import xml.etree.ElementTree as ET
import numpy as np
import mujoco

from bari_sim.robot.actions import GripAction, LiftAction, MotionAction, RobotAction
from bari_sim.robot.specification import DEFAULT_ROBOT
from bari_sim.simulation import SceneRequest, Simulation
from bari_sim.simulation import scene as scene_module
from bari_sim.tasks import parse_robot_grid

PLATE_HALF_THICK = 0.0005   # 경사면 판 두께의 절반
ROBOT_COUNT = 2

def add_chamfers(xml: str, robot, robot_count: int, chamfer_m: float) -> str:
    """뒤 조각 뒤끝의 윗면 절반, 앞 조각 앞끝의 아랫면 절반에 경사면을 붙인다."""
    root = ET.fromstring(xml)
    bodies = {b.get("name"): b for b in root.iter("body")}
    rear_len, _middle, front_len = robot.segment_lengths_m
    half_h, half_w = robot.height_m / 2.0, robot.width_m / 2.0
    c = chamfer_m
    diag = math.hypot(c, half_h) / 2.0        # 경사면 길이의 절반
    angle = -math.atan2(half_h, c)            # y축 회전(라디안; compiler angle="radian")
    nx, nz = math.sin(angle), math.cos(angle) # 회전 뒤 판의 바깥 방향
    common = {"type": "box", "mass": "0", "friction": "2.0 0.005 0.0001", "condim": "4",
              "solref": "0.015 1", "solimp": "0.9 0.95 0.001", "group": "1",
              "rgba": "0.95 0.55 0.15 1"}
    for rid in range(robot_count):
        # 뒤끝 윗면: (-rear/2, 0) → (-rear/2 + c, +h/2).  아랫면 모서리는 그대로 둔다.
        rear = dict(common)
        rear.update({
            "name": f"robot_{rid}_rear_chamfer",
            "pos": (f"{-rear_len / 2.0 + c / 2.0 + PLATE_HALF_THICK * nx:.10g} 0 "
                    f"{half_h / 2.0 + PLATE_HALF_THICK * nz:.10g}"),
            "euler": f"0 {angle:.10g} 0",
            "size": f"{diag:.10g} {half_w:.10g} {PLATE_HALF_THICK:.10g}",
        })
        ET.SubElement(bodies[f"robot_{rid}_rear_body"], "geom", rear)
        # 앞끝 아랫면: (front - c, -h/2) → (front, 0).  윗면 모서리는 그대로 둔다.
        front = dict(common)
        front.update({
            "name": f"robot_{rid}_front_chamfer",
            "pos": (f"{front_len - c / 2.0 - PLATE_HALF_THICK * nx:.10g} 0 "
                    f"{-half_h / 2.0 - PLATE_HALF_THICK * nz:.10g}"),
            "euler": f"0 {angle:.10g} 0",
            "size": f"{diag:.10g} {half_w:.10g} {PLATE_HALF_THICK:.10g}",
        })
        ET.SubElement(bodies[f"robot_{rid}_front_body"], "geom", front)
    return ET.tostring(root, encoding="unicode")

_original_build = getattr(scene_module.SceneBuilder, "_original_build", scene_module.SceneBuilder.build)
scene_module.SceneBuilder._original_build = _original_build
CHAMFER_NOW = 0.0          # make_sim이 설정한다 (0이면 경사 없음)

def _build(self):
    built = _original_build(self)
    if CHAMFER_NOW > 0.0:
        built = dataclasses.replace(
            built, xml=add_chamfers(built.xml, self.robot, self.request.grid.count, CHAMFER_NOW))
    return built

scene_module.SceneBuilder.build = _build

def make_sim(chamfer_m: float) -> Simulation:
    global CHAMFER_NOW
    CHAMFER_NOW = chamfer_m
    sim = Simulation(SceneRequest(grid=parse_robot_grid(f"{ROBOT_COUNT}*1"), environment="flat"))
    CHAMFER_NOW = 0.0
    return sim

for value in (0.0, 0.005):
    s = make_sim(value)
    names = [mujoco.mj_id2name(s.model, mujoco.mjtObj.mjOBJ_GEOM, i) or "" for i in range(s.model.ngeom)]
    print(f"경사 {value * 1000:4.1f} mm: 경사면 geom {sum('chamfer' in n for n in names)}개 / 전체 {s.model.ngeom}개")
    del s; gc.collect()
print("로봇 길이", DEFAULT_ROBOT.length_m, "m / 두께", DEFAULT_ROBOT.height_m, "m")


## 시험 실행

In [ ]:
# 2) 2대 시험: 뒤 로봇(1번)이 앞 로봇(0번) 위로 올라타는지 본다.  v2: 높이 기준을 매 스텝 다시 잰다.
DURATION_S = 30.0
START_OFFSETS = (-0.01, 0.0, 0.01, 0.02)      # 뒤 로봇 출발 위치를 바꿔 4회 반복
CHAMFERS = (0.0, 0.005, 0.010)                # 경사 없음 / 5 mm / 10 mm

# 저장소 버전에 따라 STOP이 없는 열거형이 있어서, 있으면 쓰고 없으면 생략한다.
def act(motion=None, lift=None, grip=None):
    kwargs = {k: v for k, v in (("motion", motion), ("lift", lift), ("grip", grip)) if v is not None}
    return RobotAction(**kwargs)

MOTION_STOP = getattr(MotionAction, "STOP", None)
LIFT_STOP = getattr(LiftAction, "STOP", None)
GRIP_STOP = getattr(GripAction, "STOP", None)

GAIT = [
    act(MotionAction.CURL_BODY, LiftAction.LIFT_FRONT, GRIP_STOP),
    act(MotionAction.CURL_BODY, LIFT_STOP, GRIP_STOP),
    act(MotionAction.FLATTEN_BODY, LiftAction.UNLIFT_FRONT, GRIP_STOP),
    act(MotionAction.FLATTEN_BODY, LIFT_STOP, GRIP_STOP),
]
HOLD = act(MOTION_STOP, LIFT_STOP, GRIP_STOP)
HALF_H = DEFAULT_ROBOT.height_m / 2.0

def tip(sim, rid):
    """앞 조각 끝점의 (x, z).  몸이 기울어도 맞도록 회전을 반영한다."""
    bid = sim.model.body(f"robot_{rid}_front_body").id
    mat = sim.data.xmat[bid].reshape(3, 3)
    pos = sim.data.xpos[bid]
    return (float(pos[0] + mat[0, 0] * DEFAULT_ROBOT.front_length_m),
            float(pos[2] + mat[2, 0] * DEFAULT_ROBOT.front_length_m))

def top_surface(sim, rid):
    """그 로봇 등(윗면)의 현재 높이와, 뒤끝 x."""
    ids = [sim.model.body(f"robot_{rid}_{link}_body").id for link in ("rear", "middle", "front")]
    top = max(float(sim.data.xpos[i][2]) for i in ids) + HALF_H
    rear_end = float(sim.data.xpos[ids[0]][0]) - DEFAULT_ROBOT.rear_length_m / 2.0
    return top, rear_end

def run_trial(sim, offset):
    sim.reset()
    adr = int(sim.model.jnt_qposadr[sim.model.joint("robot_1_root").id])
    sim.data.qpos[adr] += offset
    mujoco.mj_forward(sim.model, sim.data)
    rear1 = sim.model.body("robot_1_rear_body").id
    mid1 = sim.model.body("robot_1_middle_body").id
    start_x = float(sim.data.xpos[rear1][0])
    best = dict(tip_rise=-9.9, climb_len=0.0, body_rise=-9.9, on_top=False)
    step = 0
    while sim.time_s + 1e-9 < DURATION_S:
        sim.step({0: HOLD, 1: GAIT[step % len(GAIT)]})
        step += 1
        top0, rear_end0 = top_surface(sim, 0)          # 앞 로봇 등 높이(현재 자세 기준)
        tx, tz = tip(sim, 1)
        best["tip_rise"] = max(best["tip_rise"], tz - top0)          # 앞 끝이 등보다 얼마나 위인가
        if tz > top0 - 0.001:                                         # 등 높이에 도달했으면
            best["climb_len"] = max(best["climb_len"], tx - rear_end0)   # 얼마나 깊이 들어갔는가
        body_rise = float(sim.data.xpos[mid1][2]) - HALF_H - top0     # 가운데 조각 밑면 - 등 높이
        best["body_rise"] = max(best["body_rise"], body_rise)
        if body_rise > -0.001 and tx > rear_end0 + 0.02:
            best["on_top"] = True                                     # 몸통까지 등 위로 올라감
    best["advance"] = float(sim.data.xpos[rear1][0]) - start_x
    return best

results = {}
for chamfer in CHAMFERS:
    sim = make_sim(chamfer)
    tag = "경사 없음" if chamfer == 0 else f"경사 {chamfer * 1000:.0f}mm"
    rows = []
    for offset in START_OFFSETS:
        t0 = time.perf_counter()
        r = run_trial(sim, offset)
        rows.append(r)
        print(f"[{tag:9s}] 출발 {offset * 100:+5.1f}cm | 전진 {r['advance'] * 100:6.2f}cm | "
              f"앞끝-등 높이차 {r['tip_rise'] * 1000:+6.2f}mm | 등 위로 들어간 길이 {r['climb_len'] * 100:5.2f}cm | "
              f"몸통-등 높이차 {r['body_rise'] * 1000:+6.2f}mm | 올라탐 {r['on_top']} "
              f"({time.perf_counter() - t0:.0f}s)", flush=True)
    results[chamfer] = rows
    del sim; gc.collect()


## 요약

In [ ]:
# 3) 요약
print(f"{'설정':10s} {'올라탐':>7s} {'평균 전진(cm)':>13s} {'앞끝-등 높이차(mm)':>18s} {'등 위 길이(cm)':>14s}")
for chamfer, rs in results.items():
    tag = "경사 없음" if chamfer == 0 else f"경사 {chamfer * 1000:.0f}mm"
    print(f"{tag:10s} {sum(r['on_top'] for r in rs):>4d}/{len(rs)} "
          f"{np.mean([r['advance'] for r in rs]) * 100:>13.2f} "
          f"{np.mean([r['tip_rise'] for r in rs]) * 1000:>18.2f} "
          f"{np.mean([r['climb_len'] for r in rs]) * 100:>14.2f}")
print("""
읽는 법
  앞끝-등 높이차: 0보다 크면 앞 끝이 상대 등 높이까지 올라간 것 (음수면 아직 턱 아래)
  등 위 길이   : 앞 끝이 상대 등 위로 얼마나 깊이 들어갔는지 (0이면 못 올라감)
  전진         : 경사 없음보다 많이 줄었다면 경사면이 걸음을 방해하는 것""")


## 눈으로 확인 (선택)

In [ ]:
# 4) (선택) 눈으로 보기 — 경사 5 mm 설정으로 재생한다. 창을 닫으면 끝난다.
import mujoco.viewer

sim = make_sim(0.005)
sim.reset()
with mujoco.viewer.launch_passive(sim.model, sim.data) as viewer:
    step = 0
    while viewer.is_running() and sim.time_s < 30.0:
        sim.step({0: HOLD, 1: GAIT[step % len(GAIT)]},
                 frame_callback=lambda s: viewer.sync(), render_hz=60.0, realtime=True)
        step += 1
del sim; gc.collect()
